# Семинар 5. От исследования данных к готовому решению

Завершаем работу с ценами домов Ames. В предыдущих семинарах мы обучили линейную регрессию, подготовили признаки, подобрали регуляризацию и исследовали ошибки. Теперь проверим, какие доработки помогают, выберем одну модель и **впервые оценим её на финальной тестовой выборке**.

**Путь работы:** вопрос заказчика → исследование признаков → гипотезы → сравнение моделей → разбор ошибок → финальный test → записка заказчику и сохранённый Pipeline.

Это законченный учебный пример исследования. Код подготовки и графиков уже написан. Наша работа — понимать решения и проверять, следуют ли выводы из результатов. Исследование удобно разбирать в два прохода: до выбора модели и после него.

Мы будем возвращаться к знакомым понятиям: выборка, изменчивость, корреляция, матрица признаков, линейная комбинация, мультиколлинеарность, регуляризация и остатки.

## 1. Что требуется от модели?

Представим заказчика, которому нужна предварительная оценка цены дома по его характеристикам. У нас исторические сделки Ames за 2006–2010 годы. **Целевая переменная — фактическая цена продажи `SalePrice`, в долларах.** Одна строка — одна запись о продаже дома.

Для учебной постановки считаем, что характеристики дома и месяц/год оценки доступны, а условия будущей сделки ещё неизвестны. Поля `MoSold`/`YrSold` в исходном наборе означают фактическую дату продажи; здесь используем их как заданный период оценки. Если дата будущей сделки заранее неизвестна, это допущение нужно пересмотреть. `SaleType` и `SaleCondition` исследуем, но в итоговую модель не включаем. `Id` — номер записи, а не характеристика дома.

Основная метрика сравнения — **MAPE**, в процентах. Дополнительно считаем MAE и RMSE в долларах и R². MAPE придаёт больший вес ошибкам на дешёвых домах; RMSE помогает не пропустить крупные ошибки в долларах. У всех цен здесь положительные значения.

Ориентир «около 8% MAPE» из предыдущего опыта — гипотеза о достижимом качестве. Это не согласованное требование заказчика и не обещание результата. Средняя ошибка 8% также не означает, что ошибка каждого прогноза меньше 8%.

> **До финальной проверки** фиксируем вариант модели и правила обработки. По результату test описываем качество и ограничения. Если после этого меняем решение, для его окончательной проверки понадобятся новые независимые данные.

Область, которую проверяет наше случайное разбиение, — похожие исторические сделки из этого набора. Оно не подтверждает качество в другом городе или на сегодняшнем рынке.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import io
import json
import hashlib
import platform
import subprocess
import sys
import zipfile
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import cloudpickle
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.inspection import permutation_importance

sns.set_theme(style="darkgrid")
pd.set_option("display.max_colwidth", 80)
ARTIFACTS = Path("ames_final_artifacts")
ARTIFACTS.mkdir(exist_ok=True)

def metrics(y, prediction):
    return {
        "MAPE, %": 100 * mean_absolute_percentage_error(y, prediction),
        "MAE, $": mean_absolute_error(y, prediction),
        "RMSE, $": np.sqrt(mean_squared_error(y, prediction)),
        "R²": r2_score(y, prediction),
    }

def save_figure(name):
    plt.gcf().savefig(ARTIFACTS / name, dpi=130, bbox_inches="tight")

### Загружаем тот же набор и сохраняем прежнее разбиение

Теперь нам нужны все признаки. Есть небольшая, но важная поправка загрузки: строка `None` в `MasVnrType` означает **отсутствие облицовки**, а `NA` — пропуск. Обычный `read_csv` превращает оба значения в `NaN`. Сохраним различие явно.

Площади в исходном файле записаны в квадратных футах. Для графиков переводим **все** площади в м². Та же функция войдёт в Pipeline: на вход сохранённой модели можно будет подать строки в исходном формате файла.

In [ ]:
DATA_URL = (
    "https://www.dropbox.com/scl/fi/65u0zpc8dweb9rx1tlzol/train_sem2.csv"
    "?rlkey=5vi4mt47fs6magpldgj5wp2u9&dl=1"
)
csv_bytes = urlopen(DATA_URL).read()
dataset_sha256 = hashlib.sha256(csv_bytes).hexdigest()
raw_data = pd.read_csv(io.BytesIO(csv_bytes), keep_default_na=False, na_values=["NA", ""])

AREA_COLUMNS = [
    "LotArea", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea", "GarageArea", "WoodDeckSF",
    "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "PoolArea",
]

def to_metric_units(frame):
    result = frame.copy()
    columns = [c for c in AREA_COLUMNS if c in result]
    result[columns] = result[columns] * 0.09290304
    if "LotFrontage" in result:
        result["LotFrontage"] = result["LotFrontage"] * 0.3048
    return result

train_val, test = train_test_split(raw_data, test_size=0.2, random_state=42)
train, validation = train_test_split(train_val, test_size=0.25, random_state=42)
train, validation, test = train.copy(), validation.copy(), test.copy()
X_train = train.drop(columns=["Id", "SalePrice"])
y_train = train["SalePrice"]
X_val = validation.drop(columns=["Id", "SalePrice"])
y_val = validation["SalePrice"]
train_view = to_metric_units(train)
print(f"Train: {len(train)}; validation: {len(validation)}; test: {len(test)}.")
print("Test пока закрыт. Предыдущие 876/292/292 записи сохранены.")

## 2. Какие сведения о доме мы ещё не использовали?

Раньше модель видела пять числовых признаков и район. В исходном файле **79 потенциальных признаков**, не считая `Id` и целевой переменной. Среди них — качество отделки, состояние, комнаты, подвал, гараж, участок и условия продажи.

Не будем считать каждый целочисленный столбец количественным. Например, `MSSubClass` — **код типа здания**. Умножать такой код на один коэффициент означало бы приписать числам порядок и расстояния, которых в определении признака нет. Для него используем OneHot. Месяц `MoSold` также представим категориями.

Полная таблица ниже охватывает все признаки. В ней есть описание, пропуски, число разных значений и доля самого частого значения. Большая доля одной категории — повод посмотреть на оставшиеся наблюдения; сама по себе она ещё не доказывает бесполезность признака.

In [ ]:
#@title Описания признаков из словаря Ames
FEATURE_LABELS = {'MSSubClass': {'feature': 'MSSubClass', 'label': 'Тип жилого строения: код', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Дом, конструкция и качество', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'MSZoning': {'feature': 'MSZoning', 'label': 'Зонирование участка', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Участок и расположение', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'LotFrontage': {'feature': 'LotFrontage', 'label': 'Длина участка вдоль улицы', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Участок и расположение', 'unit': 'м', 'original_unit': 'ft', 'missing_kind': 'unknown', 'missing_note': 'Неизвестное значение; отсутствие объекта из этого не следует.'}, 'LotArea': {'feature': 'LotArea', 'label': 'Площадь участка', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Участок и расположение', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'Street': {'feature': 'Street', 'label': 'Покрытие подъездной улицы', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Участок и расположение', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'Alley': {'feature': 'Alley', 'label': 'Покрытие подъездного переулка', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Участок и расположение', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'structural_absence', 'missing_note': 'Нет подъездного переулка. Не путать с неизвестным значением.'}, 'LotShape': {'feature': 'LotShape', 'label': 'Форма участка', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Участок и расположение', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'LandContour': {'feature': 'LandContour', 'label': 'Рельеф участка', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Участок и расположение', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'Utilities': {'feature': 'Utilities', 'label': 'Доступные инженерные коммуникации', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Участок и расположение', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'LotConfig': {'feature': 'LotConfig', 'label': 'Расположение участка', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Участок и расположение', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'LandSlope': {'feature': 'LandSlope', 'label': 'Уклон участка', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Участок и расположение', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'Neighborhood': {'feature': 'Neighborhood', 'label': 'Район', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Участок и расположение', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'Condition1': {'feature': 'Condition1', 'label': 'Внешние условия рядом с домом', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Участок и расположение', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'Condition2': {'feature': 'Condition2', 'label': 'Дополнительные внешние условия', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Участок и расположение', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'BldgType': {'feature': 'BldgType', 'label': 'Тип жилого объекта', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Дом, конструкция и качество', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'HouseStyle': {'feature': 'HouseStyle', 'label': 'Стиль и этажность дома', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Дом, конструкция и качество', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'OverallQual': {'feature': 'OverallQual', 'label': 'Общее качество материалов и отделки', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Дом, конструкция и качество', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'OverallCond': {'feature': 'OverallCond', 'label': 'Общее состояние дома', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Дом, конструкция и качество', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'YearBuilt': {'feature': 'YearBuilt', 'label': 'Год постройки', 'kind': 'calendar_numeric', 'kind_ru': 'год', 'group': 'Дом, конструкция и качество', 'unit': 'год', 'original_unit': 'год', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'YearRemodAdd': {'feature': 'YearRemodAdd', 'label': 'Год ремонта или пристройки', 'kind': 'calendar_numeric', 'kind_ru': 'год', 'group': 'Дом, конструкция и качество', 'unit': 'год', 'original_unit': 'год', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'RoofStyle': {'feature': 'RoofStyle', 'label': 'Форма крыши', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Дом, конструкция и качество', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'RoofMatl': {'feature': 'RoofMatl', 'label': 'Материал кровли', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Дом, конструкция и качество', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'Exterior1st': {'feature': 'Exterior1st', 'label': 'Основной материал наружной обшивки', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Дом, конструкция и качество', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'Exterior2nd': {'feature': 'Exterior2nd', 'label': 'Дополнительный материал наружной обшивки', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Дом, конструкция и качество', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'MasVnrType': {'feature': 'MasVnrType', 'label': 'Тип каменной облицовки', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Дом, конструкция и качество', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'absence_and_unknown', 'missing_note': "Строка 'None' означает отсутствие облицовки; 'NA' — неизвестное значение. pd.read_csv по умолчанию объединяет их в NaN. Есть противоречия с площадью облицовки."}, 'MasVnrArea': {'feature': 'MasVnrArea', 'label': 'Площадь каменной облицовки', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Дом, конструкция и качество', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'unknown', 'missing_note': 'Неизвестное значение; отсутствие объекта из этого не следует.'}, 'ExterQual': {'feature': 'ExterQual', 'label': 'Качество наружных материалов', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Дом, конструкция и качество', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'ExterCond': {'feature': 'ExterCond', 'label': 'Состояние наружных материалов', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Дом, конструкция и качество', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'Foundation': {'feature': 'Foundation', 'label': 'Тип фундамента', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Дом, конструкция и качество', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'BsmtQual': {'feature': 'BsmtQual', 'label': 'Категория высоты подвала', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Подвал', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'structural_absence', 'missing_note': 'Нет подвала. Не путать с неизвестным значением.'}, 'BsmtCond': {'feature': 'BsmtCond', 'label': 'Состояние подвала', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Подвал', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'structural_absence', 'missing_note': 'Нет подвала. Не путать с неизвестным значением.'}, 'BsmtExposure': {'feature': 'BsmtExposure', 'label': 'Открытость стен подвала наружу', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Подвал', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'structural_absence', 'missing_note': 'Нет подвала. Не путать с неизвестным значением.'}, 'BsmtFinType1': {'feature': 'BsmtFinType1', 'label': 'Качество отделки подвала: часть 1', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Подвал', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'structural_absence', 'missing_note': 'Нет подвала. Не путать с неизвестным значением.'}, 'BsmtFinSF1': {'feature': 'BsmtFinSF1', 'label': 'Площадь отделанной части подвала 1', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Подвал', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'BsmtFinType2': {'feature': 'BsmtFinType2', 'label': 'Качество отделки подвала: часть 2', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Подвал', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'structural_absence', 'missing_note': 'Нет подвала. Не путать с неизвестным значением.'}, 'BsmtFinSF2': {'feature': 'BsmtFinSF2', 'label': 'Площадь отделанной части подвала 2', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Подвал', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'BsmtUnfSF': {'feature': 'BsmtUnfSF', 'label': 'Площадь неотделанного подвала', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Подвал', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'TotalBsmtSF': {'feature': 'TotalBsmtSF', 'label': 'Общая площадь подвала', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Подвал', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'Heating': {'feature': 'Heating', 'label': 'Тип отопления', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Инженерные системы и камины', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'HeatingQC': {'feature': 'HeatingQC', 'label': 'Качество и состояние отопления', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Инженерные системы и камины', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'CentralAir': {'feature': 'CentralAir', 'label': 'Центральное кондиционирование', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Инженерные системы и камины', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'Electrical': {'feature': 'Electrical', 'label': 'Тип электропроводки', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Инженерные системы и камины', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'unknown', 'missing_note': 'Неизвестное значение; отсутствие объекта из этого не следует.'}, '1stFlrSF': {'feature': '1stFlrSF', 'label': 'Площадь первого этажа', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Жилые площади и комнаты', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, '2ndFlrSF': {'feature': '2ndFlrSF', 'label': 'Площадь второго этажа', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Жилые площади и комнаты', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'LowQualFinSF': {'feature': 'LowQualFinSF', 'label': 'Площадь с низким качеством отделки', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Жилые площади и комнаты', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'GrLivArea': {'feature': 'GrLivArea', 'label': 'Надземная жилая площадь', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Жилые площади и комнаты', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'BsmtFullBath': {'feature': 'BsmtFullBath', 'label': 'Полные санузлы в подвале', 'kind': 'count_numeric', 'kind_ru': 'числовой счётный', 'group': 'Подвал', 'unit': 'шт.', 'original_unit': 'шт.', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'BsmtHalfBath': {'feature': 'BsmtHalfBath', 'label': 'Неполные санузлы в подвале', 'kind': 'count_numeric', 'kind_ru': 'числовой счётный', 'group': 'Подвал', 'unit': 'шт.', 'original_unit': 'шт.', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'FullBath': {'feature': 'FullBath', 'label': 'Полные санузлы надземной части', 'kind': 'count_numeric', 'kind_ru': 'числовой счётный', 'group': 'Жилые площади и комнаты', 'unit': 'шт.', 'original_unit': 'шт.', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'HalfBath': {'feature': 'HalfBath', 'label': 'Неполные санузлы надземной части', 'kind': 'count_numeric', 'kind_ru': 'числовой счётный', 'group': 'Жилые площади и комнаты', 'unit': 'шт.', 'original_unit': 'шт.', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'BedroomAbvGr': {'feature': 'BedroomAbvGr', 'label': 'Спальни надземной части', 'kind': 'count_numeric', 'kind_ru': 'числовой счётный', 'group': 'Жилые площади и комнаты', 'unit': 'шт.', 'original_unit': 'шт.', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'KitchenAbvGr': {'feature': 'KitchenAbvGr', 'label': 'Кухни надземной части', 'kind': 'count_numeric', 'kind_ru': 'числовой счётный', 'group': 'Жилые площади и комнаты', 'unit': 'шт.', 'original_unit': 'шт.', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'KitchenQual': {'feature': 'KitchenQual', 'label': 'Качество кухни', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Жилые площади и комнаты', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'TotRmsAbvGrd': {'feature': 'TotRmsAbvGrd', 'label': 'Комнаты надземной части, кроме санузлов', 'kind': 'count_numeric', 'kind_ru': 'числовой счётный', 'group': 'Жилые площади и комнаты', 'unit': 'шт.', 'original_unit': 'шт.', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'Functional': {'feature': 'Functional', 'label': 'Пригодность дома к эксплуатации', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Дом, конструкция и качество', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'Fireplaces': {'feature': 'Fireplaces', 'label': 'Количество каминов', 'kind': 'count_numeric', 'kind_ru': 'числовой счётный', 'group': 'Инженерные системы и камины', 'unit': 'шт.', 'original_unit': 'шт.', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'FireplaceQu': {'feature': 'FireplaceQu', 'label': 'Качество камина', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Инженерные системы и камины', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'structural_absence', 'missing_note': 'Нет камина. Не путать с неизвестным значением.'}, 'GarageType': {'feature': 'GarageType', 'label': 'Расположение гаража', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Гараж', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'structural_absence', 'missing_note': 'Нет гаража. Не путать с неизвестным значением.'}, 'GarageYrBlt': {'feature': 'GarageYrBlt', 'label': 'Год постройки гаража', 'kind': 'calendar_numeric', 'kind_ru': 'год', 'group': 'Гараж', 'unit': 'год', 'original_unit': 'год', 'missing_kind': 'structural_absence', 'missing_note': 'Нет гаража. Для GarageYrBlt отсутствие согласовано с нулевыми GarageCars/GarageArea.'}, 'GarageFinish': {'feature': 'GarageFinish', 'label': 'Степень отделки гаража', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Гараж', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'structural_absence', 'missing_note': 'Нет гаража. Не путать с неизвестным значением.'}, 'GarageCars': {'feature': 'GarageCars', 'label': 'Вместимость гаража в автомобилях', 'kind': 'count_numeric', 'kind_ru': 'числовой счётный', 'group': 'Гараж', 'unit': 'шт.', 'original_unit': 'шт.', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'GarageArea': {'feature': 'GarageArea', 'label': 'Площадь гаража', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Гараж', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'GarageQual': {'feature': 'GarageQual', 'label': 'Качество гаража', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Гараж', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'structural_absence', 'missing_note': 'Нет гаража. Не путать с неизвестным значением.'}, 'GarageCond': {'feature': 'GarageCond', 'label': 'Состояние гаража', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Гараж', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'structural_absence', 'missing_note': 'Нет гаража. Не путать с неизвестным значением.'}, 'PavedDrive': {'feature': 'PavedDrive', 'label': 'Покрытие подъездной дорожки', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Дополнительные объекты', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'WoodDeckSF': {'feature': 'WoodDeckSF', 'label': 'Площадь деревянной террасы', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Дополнительные объекты', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'OpenPorchSF': {'feature': 'OpenPorchSF', 'label': 'Площадь открытого крыльца', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Дополнительные объекты', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'EnclosedPorch': {'feature': 'EnclosedPorch', 'label': 'Площадь закрытого крыльца', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Дополнительные объекты', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, '3SsnPorch': {'feature': '3SsnPorch', 'label': 'Площадь трёхсезонной веранды', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Дополнительные объекты', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'ScreenPorch': {'feature': 'ScreenPorch', 'label': 'Площадь веранды с защитной сеткой', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Дополнительные объекты', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'PoolArea': {'feature': 'PoolArea', 'label': 'Площадь бассейна', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Дополнительные объекты', 'unit': 'м²', 'original_unit': 'ft²', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'PoolQC': {'feature': 'PoolQC', 'label': 'Качество бассейна', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Дополнительные объекты', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'structural_absence', 'missing_note': 'Нет бассейна. Не путать с неизвестным значением.'}, 'Fence': {'feature': 'Fence', 'label': 'Тип и качество забора', 'kind': 'ordinal', 'kind_ru': 'порядковый', 'group': 'Дополнительные объекты', 'unit': 'уровень', 'original_unit': 'уровень', 'missing_kind': 'structural_absence', 'missing_note': 'Нет забора. Не путать с неизвестным значением.'}, 'MiscFeature': {'feature': 'MiscFeature', 'label': 'Дополнительный объект на участке', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Дополнительные объекты', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'structural_absence', 'missing_note': 'Нет дополнительного объекта. Не путать с неизвестным значением.'}, 'MiscVal': {'feature': 'MiscVal', 'label': 'Стоимость дополнительного объекта', 'kind': 'continuous_numeric', 'kind_ru': 'числовой непрерывный', 'group': 'Дополнительные объекты', 'unit': '$', 'original_unit': '$', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'MoSold': {'feature': 'MoSold', 'label': 'Месяц продажи', 'kind': 'calendar_category', 'kind_ru': 'календарная категория', 'group': 'Продажа', 'unit': 'месяц', 'original_unit': 'месяц', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'YrSold': {'feature': 'YrSold', 'label': 'Год продажи', 'kind': 'calendar_numeric', 'kind_ru': 'год', 'group': 'Продажа', 'unit': 'год', 'original_unit': 'год', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'SaleType': {'feature': 'SaleType', 'label': 'Тип сделки', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Продажа', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}, 'SaleCondition': {'feature': 'SaleCondition', 'label': 'Условия сделки', 'kind': 'nominal', 'kind_ru': 'номинальный', 'group': 'Продажа', 'unit': 'категория', 'original_unit': 'категория', 'missing_kind': 'none_in_train', 'missing_note': 'В train пропусков нет.'}}

In [ ]:
def feature_label(name):
    value = FEATURE_LABELS.get(name, name)
    if isinstance(value, dict):
        return value.get("label", value.get("description", name))
    return str(value)

inventory_rows = []
for name in X_train:
    values = train[name]
    counts = values.fillna("<пропуск>").value_counts()
    is_category = not pd.api.types.is_numeric_dtype(values) or name in ["MSSubClass", "MoSold"]
    inventory_rows.append({
        "Признак": name, "Описание": feature_label(name),
        "Тип для модели": "категориальный" if is_category else "числовой",
        "Пропусков": int(values.isna().sum()), "Разных значений": values.nunique(),
        "Самое частое": str(counts.index[0]), "Его доля, %": 100 * counts.iloc[0] / len(values),
        "Редких категорий (<10)": int((counts < 10).sum()) if is_category else np.nan,
        "Решение": "только исследование: условия будущей сделки" if name in ["SaleType", "SaleCondition"] else "кандидат в модель",
    })
inventory = pd.DataFrame(inventory_rows)
inventory.to_csv(ARTIFACTS / "feature_inventory.csv", index=False)
with pd.option_context("display.max_rows", 90):
    display(inventory.round(1))

### Пропуск не всегда означает неизвестное значение

| Пример | Что означает запись | Как поступаем |
|---|---|---|
| `LotFrontage = NA` | Длина фасада участка неизвестна | Заполняем медианой тренировочной части |
| `MasVnrType = None` | Облицовки нет | Оставляем отдельную категорию |
| `MasVnrType = NA` | Тип облицовки неизвестен | Отдельная категория «неизвестно» |
| `GarageQual = NA`, площадь гаража равна 0 | Гаража нет | Отдельная категория «нет объекта» |
| `GarageQual = NA`, площадь гаража положительна | Гараж есть, качество неизвестно | Отдельная категория «неизвестно» |

Это восстановление смысла данных по словарю, а не подбор заполнения по цене дома. Для числового года гаража без гаража используем медианное заполнение; отдельные категории отсутствия гаража позволяют модели различать эти случаи. В более подробном исследовании это правило можно сравнить с отдельным индикатором.

В тренировочной выборке у `MasVnrType` 499 записей `None` и только 5 неизвестных значений. Их смешение скрыло бы полезное различие. Встречаются и противоречивые записи: тип облицовки «нет», но площадь положительна. Без проверки источника не будем угадывать, какое поле верно.

[Словарь Ames, De Cock](https://jse.amstat.org/v19n3/decock/DataDocumentation.txt) · [Описание набора и учебных особенностей](https://jse.amstat.org/v19n3/decock.pdf)

### Площадь, качество и год постройки

Снова посмотрим на скаттерплоты. Цвет помогает увидеть, что зависимость цены от года постройки может одновременно отражать различия в качестве домов. **Корреляция не позволяет отделить причины друг от друга.** Это знакомая задача интерпретации при скоррелированных признаках.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, feature, label in zip(axs, ["GrLivArea", "OverallQual", "YearBuilt"],
                            ["Жилая площадь, м²", "Общее качество, баллы 1–10", "Год постройки"]):
    sns.scatterplot(data=train_view, x=feature, y="SalePrice", hue="OverallQual",
                    palette="viridis", alpha=0.55, ax=ax, legend=(feature == "YearBuilt"))
    ax.set(xlabel=label, ylabel="Цена продажи, $", title="Тренировочная выборка")
axs[2].legend(title="Качество", fontsize=8)
fig.tight_layout()
save_figure("01_train_relationships.png")
plt.show()

display(train[["YearBuilt", "OverallQual", "SalePrice"]].corr().round(3))
year_groups = train.assign(Период=np.where(train.YearBuilt < 1990, "До 1990", "С 1990"))
display(year_groups.groupby("Период").agg(
    Число_домов=("SalePrice", "size"), Медиана_цены=("SalePrice", "median"),
    Медиана_качества=("OverallQual", "median")))

**Что заметили.** В train корреляция года постройки с ценой около 0,51, а с качеством — около 0,54. Корреляция качества с ценой около 0,78. Год постройки связан с другими характеристиками дома, поэтому видимую на графике форму нельзя целиком приписывать возрасту.

Проверим несколько гипотез: добавить сведения о качестве; изменить форму зависимости от площади; сравнить непрерывный год с индикатором «построен с 1990 года». **Порог 1990 — кандидат для эксперимента.** Разница средних или медиан двух групп сама по себе не доказывает, что такая замена улучшит прогноз.

Два очень больших дома имеют неожиданно низкую цену. Мы уже знаем: необычная точка может сильно влиять на обучение и метрики. Пока сохраняем все наблюдения; удаление по принципу «мешает получить красивую метрику» не является проверкой данных.

## 3. Категории: различия в цене и число наблюдений

Для категории считаем число домов, медиану цены и квартили. Если в группе два дома, её медиану нельзя интерпретировать так же уверенно, как медиану группы из двухсот домов.

Похожие медианы **не доказывают одинакового влияния категорий**: группы могут различаться площадью, качеством и другими признаками. Поэтому не станем автоматически объединять категории по цене. Проверим простое правило: редкие категории объединяются по числу наблюдений в тренировочной части, без использования целевой переменной.

In [ ]:
categorical_columns = inventory.loc[inventory["Тип для модели"] == "категориальный", "Признак"].tolist()
category_tables = []
for column in categorical_columns:
    table = train.assign(_category=train[column].fillna("<пропуск>").astype(str)).groupby("_category").agg(
        n=("SalePrice", "size"), median_price=("SalePrice", "median"),
        q1=("SalePrice", lambda s: s.quantile(.25)), q3=("SalePrice", lambda s: s.quantile(.75)))
    table = table.reset_index().rename(columns={"_category": "category"})
    table.insert(0, "feature", column)
    category_tables.append(table)
category_statistics = pd.concat(category_tables, ignore_index=True)
category_statistics.to_csv(ARTIFACTS / "category_statistics.csv", index=False)

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
for ax, column in zip(axs, ["Neighborhood", "KitchenQual"]):
    table = category_statistics.query("feature == @column").sort_values("median_price")
    positions = np.arange(len(table))
    ax.errorbar(table.median_price, positions,
                xerr=np.vstack([table.median_price-table.q1, table.q3-table.median_price]),
                fmt="o", capsize=3)
    ax.set_yticks(positions, [f"{r.category} · n={r.n}" for r in table.itertuples()])
    ax.set(xlabel="Цена продажи, $: медиана и интервал Q1–Q3",
           ylabel="Район" if column == "Neighborhood" else "Качество кухни",
           title=f"{column}: тренировочная выборка")
fig.tight_layout()
save_figure("02_categories.png")
plt.show()

**Как прочитать результат.** У районов отличаются и медианы, и разброс цен. В качестве кухни категории `Ex`, `Gd`, `TA`, `Fa` задают порядок, но расстояния между ними в цене не обязаны быть одинаковыми.

Сравним два знакомых представления: отдельный OneHot-столбец на категорию и числовую шкалу качества. OneHot даёт каждой категории свой вес; шкала использует один вес и предполагает одинаковый шаг. Более компактное представление может оказаться полезным, но это проверяется на отложенных наблюдениях.

Для редких категорий используем `OneHotEncoder(min_frequency=10, handle_unknown="infrequent_if_exist")`. Частоты и состав объединённой группы определяются при `fit`. Если редкой группы не возникло, новая категория кодируется нулями; это техническое правило, а не доказательство её равенства известным категориям. Сравним вариант с объединением и без него.

[OneHotEncoder: редкие и неизвестные категории](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)

## 4. Меняем признаки, сохраняя линейную модель

Вспомним: прогноз линейной модели — **линейная комбинация столбцов матрицы признаков**. Столбцом может быть исходная площадь, её логарифм, квадрат качества или произведение двух признаков.

В исходных данных некоторые столбцы точно выражаются через другие: жилая площадь равна сумме площадей первого этажа, второго этажа и жилых помещений низкого качества. Это **точная линейная зависимость**. Расширение набора признаков не всегда добавляет новую информацию; мы уже встречались с этим при обсуждении ранга матрицы и Ridge.

$$\widehat y=w_0+w_1x+w_2x^2.$$

Здесь $x$ — один исходный признак; $x^2$ — дополнительный признак; $w_0$ — свободный член; $w_1,w_2$ — веса; $\widehat y$ — прогноз. Зависимость от $x$ нелинейна, но модель по-прежнему линейна по весам. Добавление похожих столбцов может усиливать **мультиколлинеарность** — поэтому регуляризация остаётся полезной.

Проверим также логарифмирование площади и целевой переменной:

$$z=\ln(1+y),\qquad \widehat z=w_0+\sum_{j=1}^{p}w_j\phi_j(x),\qquad \widehat y=\exp(\widehat z)-1.$$

Здесь $y$ — числовое значение цены в долларах; $z$ — преобразованная целевая переменная; $x$ — описание дома; $\phi_j(x)$ — исходный или преобразованный признак; $p$ — число столбцов; $w_j$ — веса; $\widehat z$ и $\widehat y$ — прогноз до и после обратного преобразования.

На логарифмической шкале ошибки на дорогих домах не доминируют в сумме квадратов так сильно. Но мы меняем задачу оптимизации: Ridge минимизирует квадратичную ошибку **логарифма цены** с регуляризацией. Это не прямое обучение по MAPE. После обратного преобразования проверяем все метрики в исходных единицах.

Обратное преобразование среднего логарифма не равно среднему цены. Поэтому такое изменение может улучшить относительные ошибки и одновременно изменить систематическое смещение. Проверим это по остаткам.

`TransformedTargetRegressor` выполняет преобразование цели при `fit` и обратное преобразование при `predict` внутри общей цепочки. [Пример sklearn на Ames](https://scikit-learn.org/stable/auto_examples/compose/plot_transformed_target.html)

In [ ]:
#@title Подготовленная цепочка: единицы → признаки → заполнение/кодирование → регрессия
SIX = ['GrLivArea','GarageArea','TotalBsmtSF','YearBuilt','LotFrontage','Neighborhood']
QUALITY = ['OverallQual','OverallCond','ExterQual','KitchenQual','YearRemodAdd','GarageCars']
ABSENT = ['Alley','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2',
          'FireplaceQu','GarageType','GarageFinish','GarageQual','GarageCond','PoolQC','Fence','MiscFeature']
QUALITY_MAP = {'Po':1.,'Fa':2.,'TA':3.,'Gd':4.,'Ex':5.,'__ABSENT__':0.}
ORDINAL = ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC','KitchenQual',
           'FireplaceQu','GarageQual','GarageCond','PoolQC']

class Features(BaseEstimator, TransformerMixin):
    def __init__(self, feature_set='all', year_style='linear', ordinal=False,
                 nonlinear=False, log_features=False, remove_sale_info=False):
        self.feature_set = feature_set
        self.year_style = year_style
        self.ordinal = ordinal
        self.nonlinear = nonlinear
        self.log_features = log_features
        self.remove_sale_info = remove_sale_info

    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.feature_names_out_ = np.asarray(self.transform(X.iloc[:0]).columns, dtype=object)
        return self

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_out_

    def transform(self, X):
        z = X.drop(columns=['Id','SalePrice'], errors='ignore').copy()
        if self.feature_set == 'six':
            z = z[SIX].copy()
        elif self.feature_set == 'quality':
            z = z[SIX + QUALITY].copy()
        for c in ABSENT:
            if c not in z:
                continue
            companion = ('TotalBsmtSF' if c.startswith('Bsmt') else
                         'GarageArea' if c.startswith('Garage') else
                         'Fireplaces' if c == 'FireplaceQu' else
                         'PoolArea' if c == 'PoolQC' else None)
            if companion in z:
                absent = z[c].isna() & z[companion].eq(0)
                z.loc[absent, c] = '__ABSENT__'
            else:
                z[c] = z[c].fillna('__ABSENT__')
        for c in ['MSSubClass','MoSold']:
            if c in z:
                z[c] = z[c].astype(str)
        if self.ordinal:
            for c in ORDINAL:
                if c in z:
                    z[c] = z[c].map(QUALITY_MAP)
        if self.year_style in ['binary','both']:
            z['BuiltAfter1990'] = (z['YearBuilt'] >= 1990).astype(float)
            if self.year_style == 'binary':
                z = z.drop(columns='YearBuilt')
        if self.nonlinear:
            z['OverallQualSquared'] = z['OverallQual'] ** 2
            z['AreaQuality'] = z['GrLivArea'] * z['OverallQual']
            z['TotalArea'] = z['GrLivArea'] + z['TotalBsmtSF']
            z['TotalBath'] = z['FullBath'] + .5*z['HalfBath'] + z['BsmtFullBath'] + .5*z['BsmtHalfBath']
            z['HouseAge'] = z['YrSold'] - z['YearBuilt']
            z['RemodelAge'] = z['YrSold'] - z['YearRemodAdd']
        if self.log_features:
            for c in AREA_COLUMNS + ['LotFrontage','TotalArea']:
                if c in z:
                    z[c] = np.log1p(z[c])
        if self.remove_sale_info:
            z = z.drop(columns=['SaleType','SaleCondition'], errors='ignore')
        return z

def build_model(X, log_target=False, rare=None, estimator='ridge', **kwargs):
    transform = Features(**kwargs)
    sample = transform.transform(X.iloc[:0])
    numeric = sample.select_dtypes(include='number').columns.tolist()
    categorical = [c for c in sample if c not in numeric]
    num = Pipeline([('imputer',SimpleImputer(strategy='median', keep_empty_features=True)),
                    ('scale',StandardScaler())])
    cat = Pipeline([('imputer',SimpleImputer(strategy='constant',fill_value='__UNKNOWN__',keep_empty_features=True)),
                    ('onehot',OneHotEncoder(handle_unknown='infrequent_if_exist' if rare else 'ignore',
                                            min_frequency=rare, sparse_output=False,
                                            drop='first' if kwargs.get('feature_set')=='six' else None))])
    prep = ColumnTransformer([('num',num,numeric),('cat',cat,categorical)])
    regression = Ridge(alpha=10) if estimator=='ridge' else LinearRegression()
    if log_target:
        regression = TransformedTargetRegressor(regressor=regression,func=np.log1p,inverse_func=np.expm1)
    return Pipeline([('units', FunctionTransformer(to_metric_units)),('features',transform),('preprocess',prep),('regression',regression)])

**Что здесь уже знакомо.** `Pipeline` объединяет действия. В `ColumnTransformer` числовые признаки проходят заполнение медианой и `StandardScaler`, а категории — заполнение и OneHot. Медианы, средние, стандартные отклонения и категории запоминаются только при `fit`.

`Features` — подготовленный трансформер для наших проверяемых вариантов. Он меняет представление столбцов, но не вычисляет статистики по целевой переменной. Его длинный код не требуется переписывать вручную.

Для полной Ridge-модели оставляем все OneHot-столбцы. Они линейно зависимы со свободным членом, однако положительная L2-регуляризация позволяет оценить веса. В знакомой модели на шести признаках сохраняем прежний `drop="first"`. Коэффициенты двух кодировок нельзя напрямую сравнивать как эффекты относительно одного базового района.

## 5. Как выбрать `alpha`, не подстраиваясь под одну validation?

> **Кросс-валидация (cross-validation, CV)** — способ оценки процедуры обучения: данные несколько раз разделяют на тренировочную и проверочную части; каждый раз заново обучают всю цепочку и считают метрику на наблюдениях, не участвовавших в этом обучении.

Используем **K-fold CV**: делим прежний train на пять частей — фолдов. Четыре части служат для обучения, пятая для проверки. Повторяем пять раз, меняя проверочную часть. Прежняя validation остаётся для последующего сравнения и разбора ошибок; финальный test пока не открываем.

$$\operatorname{CV\!\text{-}\!MAPE}=\frac1K\sum_{k=1}^{K}\operatorname{MAPE}_k.$$

Здесь $K=5$ — число фолдов; $\operatorname{MAPE}_k$ — MAPE на проверочной части в повторе $k$. Это среднее пяти оценок. Их стандартное отклонение показывает изменчивость результата между частями данных; оно не является готовым доверительным интервалом.

**Гиперпараметр** `alpha` задаём до обучения весов. Проверим небольшую сетку значений, выбирая наименьшую среднюю CV-MAPE. `GridSearchCV` выполняет этот перебор. И заполнение пропусков, и масштабирование, и объединение редких категорий заново обучаются внутри каждого повтора.

Такое случайное разбиение подходит к нашей учебной постановке с похожими историческими сделками. Для прогноза будущего периода нужна проверка по времени, а для повторных записей одного объекта — разбиение с учётом групп.

В коде `neg_mean_absolute_percentage_error` возвращает отрицательную MAPE: sklearn сравнивает оценки по правилу «больше — лучше». Знак минус возвращает обычную ошибку, множитель 100 — проценты. После выбора `alpha` параметр `refit="mape"` запускает ещё одно обучение на всём train. `best_estimator_` — эта одна обученная цепочка; прогнозы пяти моделей из CV здесь не усредняются.

[Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html) · [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) · [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
folds = list(cv.split(X_train))
scheme = np.eye(5)
fig, ax = plt.subplots(figsize=(8, 3))
sns.heatmap(scheme, annot=np.where(scheme == 1, "Проверка", "Обучение"), fmt="",
            cmap=["#c8dce8", "#efb366"], cbar=False, linewidths=3, ax=ax)
ax.set(xlabel="Часть тренировочной выборки", ylabel="Повтор обучения",
       xticklabels=[1, 2, 3, 4, 5], yticklabels=[1, 2, 3, 4, 5],
       title="Каждое наблюдение один раз попадает в проверочную часть")
plt.tight_layout()
plt.show()

### Заранее задаём конечный список экспериментов

Сравним знакомую модель с расширенной. Затем проверим логарифм цели, логарифмы площадей, редкие категории, индикатор года, шкалу качества и несколько дополнительных столбцов. **Для каждого изменения есть вопрос, а не обещание улучшения.**

Вариант с дополнительными признаками добавляет квадрат качества, произведение площади на качество, суммарную площадь, число ванных и возраст дома/ремонта. Это небольшой пример feature engineering; все возможные произведения 79 признаков нам сейчас не нужны.

In [ ]:
ALPHAS = [0.1, 1, 10, 30, 100, 300]
CANDIDATES = [
    ("01", "OLS: прежние 6 признаков", dict(feature_set="six", estimator="ols")),
    ("02", "Ridge: прежние 6 признаков", dict(feature_set="six")),
    ("03", "Ridge: больше сведений о доме", dict(remove_sale_info=True)),
    ("04", "+ логарифм цены", dict(remove_sale_info=True, log_target=True)),
    ("05", "+ логарифмы площадей", dict(remove_sale_info=True, log_target=True, log_features=True)),
    ("06", "+ объединение редких категорий", dict(remove_sale_info=True, log_target=True, log_features=True, rare=10)),
    ("07", "Вариант 06: год → до/после 1990", dict(remove_sale_info=True, log_target=True, log_features=True, rare=10, year_style="binary")),
    ("08", "Вариант 06: дополнительные столбцы", dict(remove_sale_info=True, log_target=True, log_features=True, rare=10, nonlinear=True)),
    ("09", "Вариант 06: качество как шкала", dict(remove_sale_info=True, log_target=True, log_features=True, rare=10, ordinal=True)),
]
scoring = {"mape": "neg_mean_absolute_percentage_error", "mae": "neg_mean_absolute_error"}
fitted_models, searches, cv_rows = {}, {}, []

for key, title, settings in CANDIDATES:
    model = build_model(X_train, **settings)
    if settings.get("estimator") == "ols":
        scores = cross_validate(model, X_train, y_train, cv=folds, scoring=scoring)
        fold_errors = -100 * scores["test_mape"]
        fitted = model.fit(X_train, y_train)
        alpha = np.nan
    else:
        path = "regression__regressor__alpha" if settings.get("log_target") else "regression__alpha"
        search = GridSearchCV(model, {path: ALPHAS}, cv=folds, scoring=scoring,
                              refit="mape", n_jobs=1, error_score="raise")
        search.fit(X_train, y_train)
        fitted = search.best_estimator_
        alpha = search.best_params_[path]
        i = search.best_index_
        fold_errors = np.array([-100 * search.cv_results_[f"split{k}_test_mape"][i] for k in range(5)])
        searches[key] = search
    fitted_models[key] = fitted
    cv_rows.append({"Код": key, "Вариант": title, "alpha": alpha,
                    "CV MAPE, %": fold_errors.mean(), "SD между фолдами, п.п.": fold_errors.std(),
                    **{f"Фолд {i+1}, %": v for i, v in enumerate(fold_errors)}})
    print(f"Готово: {title}")

cv_results = pd.DataFrame(cv_rows).set_index("Код")
display(cv_results.round(3))
cv_results.to_csv(ARTIFACTS / "cv_comparison.csv")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
for key in ["02", "03", "06"]:
    results = pd.DataFrame(searches[key].cv_results_)
    values = [list(p.values())[0] for p in results["params"]]
    axs[0].plot(values, -100 * results["mean_test_mape"], "o-", label=cv_results.loc[key, "Вариант"])
axs[0].set(xscale="log", xlabel="Гиперпараметр alpha", ylabel="Средняя CV-MAPE, %", title="Подбираем силу регуляризации")
axs[0].legend(fontsize=8)
for key in ["01", "04", "06"]:
    axs[1].plot(range(1, 6), cv_results.loc[key, [f"Фолд {i}, %" for i in range(1, 6)]].astype(float),
                "o-", label=cv_results.loc[key, "Вариант"])
axs[1].set(xlabel="Номер проверочной части", ylabel="MAPE на части, %", title="Среднее не скрывает различия частей", xticks=range(1, 6))
axs[1].legend(fontsize=8)
fig.tight_layout()
save_figure("03_cv.png")
plt.show()

**Как принимаем решение.** Смотрим на среднюю ошибку и на отдельные фолды. Логарифмирование только цены может сделать прогнозы отдельных необычных домов ещё хуже. Поэтому правило «логарифм всегда улучшает регрессию» неверно.

Нам важна и простота решения. Почти одинаковые средние значения не доказывают преимущество более сложного варианта. Стандартное отклонение между фолдами помогает заметить чувствительность к составу выборки, но не заменяет отдельную проверку различия моделей.

Код ниже возвращает лучшие по CV настройки каждого заранее заданного варианта на прежнюю validation. Это ещё этап разработки: по этой таблице и графикам мы выбираем решение. Финальная тестовая оценка будет позже.

In [ ]:
validation_predictions = {key: model.predict(X_val) for key, model in fitted_models.items()}
validation_results = pd.DataFrame([
    {"Код": key, "Вариант": title, **metrics(y_val, validation_predictions[key])}
    for key, title, _ in CANDIDATES
]).set_index("Код")
display(validation_results.round(3))
validation_results.to_csv(ARTIFACTS / "validation_comparison.csv")

**Вывод предварительного исследования.** Дополнительные сведения о доме и логарифмы площадей дают основной выигрыш. Порог 1990 года, шкала качества и дополнительные произведения признаков не становятся обязательными только потому, что выглядят разумно.

Для завершения исследования выбираем **вариант 06**: Ridge, логарифм цены, логарифмы площадей, OneHot с объединением редких категорий; условия сделки исключены. `alpha` берём из CV. Объединение редких категорий — простое правило обработки малых групп, а не заявление, что у этих категорий одинаковые цены. Варианты 05 и 06 практически близки: разница CV-MAPE около 0,004 процентного пункта, а RMSE на validation у 06 даже немного выше. Это не существенный доказанный выигрыш от объединения категорий.

Внутри этого набора экспериментов не требуется любой ценой получить минимальное число после запятой. Далее проверяем, что именно улучшилось и какие ошибки остались. Если при другом разбиении более простой вариант устойчиво окажется лучше, выбор стоит пересмотреть **до** обращения к test.

### Нужны ли две отдельные модели?

В предыдущем семинаре мы разбивали дома на группы по площади и видели различия ошибок. Теперь проверим гипотезу о **разном наклоне зависимости цены от площади**. На графике ниже проведём отдельные прямые для двух групп. Это описательная одномерная связь: её наклон может отражать и различия качества, района и других признаков.

Для опыта фиксируем границу 150 м² — близко к прежней границе верхней трети train (около 156 м²). Не подбираем её по test. Сравним одну модель с двумя моделями по площади, используя одинаковые преобразования. Во второй схеме каждая группа получает свой набор весов. В группах меньше наблюдений, а на границе возможен скачок прогноза.

Логарифмирование площади также позволяет одной модели описывать изменение скорости зависимости. Поэтому проверяем, нужен ли поверх этого ещё и переход к двум моделям.

Для каждой группы обучаем весь Pipeline отдельно. Используем одну общую сетку `alpha` и одинаковые фолды; дополнительные варианты границы не перебираем.

In [ ]:
AREA_BOUNDARY = 150.0
fig, ax = plt.subplots(figsize=(9, 4.5))
slope_rows = []
for mask, title, color in [
    (train_view.GrLivArea < AREA_BOUNDARY, "Площадь меньше 150 м²", "tab:blue"),
    (train_view.GrLivArea >= AREA_BOUNDARY, "Площадь от 150 м²", "tab:orange"),
]:
    subset = train_view.loc[mask]
    slope, intercept = np.polyfit(subset.GrLivArea, subset.SalePrice, 1)
    grid = np.linspace(subset.GrLivArea.min(), subset.GrLivArea.max(), 100)
    ax.scatter(subset.GrLivArea, subset.SalePrice, color=color, s=16, alpha=.25)
    ax.plot(grid, intercept + slope * grid, color=color, linewidth=2, label=title)
    slope_rows.append({"Группа": title, "Число домов": len(subset), "Наклон прямой, $/м²": slope})
ax.axvline(AREA_BOUNDARY, color="black", linestyle="--", linewidth=1)
ax.set(xlabel="Жилая площадь, м²", ylabel="Цена продажи, $", title="Train: отдельные прямые в группах площади")
ax.legend()
fig.tight_layout()
save_figure("03b_area_slopes.png")
plt.show()
display(pd.DataFrame(slope_rows).round(1))

In [ ]:
#@title Сравнение общей модели с двумя моделями; те же пять фолдов
SEGMENT_ALPHAS = [1, 10, 30, 100]
segment_settings = dict(remove_sale_info=True, log_target=True, log_features=True, rare=10)
segment_rules = {"Одна модель": None, "Две модели: площадь 150 м²": ("GrLivArea", AREA_BOUNDARY)}

def train_segment_models(X, y, rule, alpha):
    group = pd.Series(0, index=X.index) if rule is None else (to_metric_units(X)[rule[0]] >= rule[1]).astype(int)
    models = {}
    for value in sorted(group.unique()):
        mask = group == value
        estimator = build_model(X, **segment_settings)
        estimator.set_params(regression__regressor__alpha=alpha)
        models[value] = estimator.fit(X.loc[mask], y.loc[mask])
    return models

def predict_segments(models, X, rule):
    group = pd.Series(0, index=X.index) if rule is None else (to_metric_units(X)[rule[0]] >= rule[1]).astype(int)
    prediction = pd.Series(index=X.index, dtype=float)
    for value, estimator in models.items():
        mask = group == value
        prediction.loc[mask] = estimator.predict(X.loc[mask])
    return prediction.to_numpy()

segment_rows = []
for title, rule in segment_rules.items():
    alpha_scores = []
    for alpha in SEGMENT_ALPHAS:
        fold_scores = []
        for fit_idx, check_idx in folds:
            models = train_segment_models(X_train.iloc[fit_idx], y_train.iloc[fit_idx], rule, alpha)
            p = predict_segments(models, X_train.iloc[check_idx], rule)
            fold_scores.append(100 * mean_absolute_percentage_error(y_train.iloc[check_idx], p))
        alpha_scores.append((np.mean(fold_scores), alpha))
    score, alpha = min(alpha_scores)
    models = train_segment_models(X_train, y_train, rule, alpha)
    prediction = predict_segments(models, X_val, rule)
    group_counts = [len(X_train)] if rule is None else (to_metric_units(X_train)[rule[0]] >= rule[1]).value_counts().sort_index().tolist()
    segment_rows.append({"Вариант": title, "Размеры train-групп": str(group_counts), "alpha": alpha,
                         "CV MAPE, %": score, **metrics(y_val, prediction)})
segment_results = pd.DataFrame(segment_rows)
display(segment_results.round(3))
segment_results.to_csv(ARTIFACTS / "segmentation_comparison.csv", index=False)

**Что проверяем в этой таблице.** Улучшение MAPE не должно скрывать рост RMSE и крупных ошибок в маленькой группе. Отдельные модели полезны, когда различия закономерностей подтверждаются проверкой и для каждой группы хватает наблюдений. В нашем итоговом решении оставляем одну модель; сами группы используем для диагностики качества.

В этом опыте разделение по площади повысило CV-MAPE примерно с 9,04% до 9,74%, а MAPE на validation — с 8,74% до 9,04%. Значит, **разные наклоны на скаттерплоте дали гипотезу, но выбранный способ её реализации не улучшил прогноз**. Это не запрещает сегментацию вообще: мы проверили конкретную границу, модели и набор данных.

## 6. Стали ли лучше прогнозы проблемных домов?

Возвращаемся к тем же пяти домам из предыдущего семинара. Добавляем в таблицу общее качество, качество кухни и категорию высоты подвала (`BsmtQual`). Сравниваем прежний Ridge на шести признаках и выбранную модель на **тех же** validation-объектах.

Это позволяет увидеть, какие ошибки сократил доработанный вариант и какие необычные объекты всё ещё плохо описываются моделью. Мы изменили и признаки, и форму зависимости; вклад отдельных шагов обсуждаем по таблице экспериментов. Улучшение общей метрики не обещает улучшения каждого дома.

In [ ]:
SELECTED = "06"
chosen_model = fitted_models[SELECTED]
old_prediction = validation_predictions["02"]
chosen_prediction = validation_predictions[SELECTED]
problem_ids = [1170, 179, 1182, 54, 1325]
cases = validation[["Id", "Neighborhood", "OverallQual", "KitchenQual", "BsmtQual", "SalePrice"]].copy()
cases["Прежний прогноз, $"] = old_prediction
cases["Новый прогноз, $"] = chosen_prediction
cases["Прежняя |ошибка|, $"] = abs(y_val - old_prediction)
cases["Новая |ошибка|, $"] = abs(y_val - chosen_prediction)
cases = cases.set_index("Id").loc[problem_ids]
display(cases.round(0))
cases.to_csv(ARTIFACTS / "five_known_cases.csv")

fig, ax = plt.subplots(figsize=(9, 4))
positions = np.arange(len(cases))
ax.bar(positions-.18, cases["Прежняя |ошибка|, $"], width=.36, label="Прежние 6 признаков")
ax.bar(positions+.18, cases["Новая |ошибка|, $"], width=.36, label="Выбранная модель")
ax.set(xticks=positions, xticklabels=cases.index, xlabel="Id дома из validation",
       ylabel="Абсолютная ошибка, $", title="Доработка модели помогает не всем одинаково")
ax.legend()
fig.tight_layout()
save_figure("04_known_errors.png")
plt.show()

**Что получилось.** У дома 1170 абсолютная ошибка сократилась примерно с 231 до 142 тысяч долларов, у дома 54 — со 130 до 88 тысяч. Улучшение есть, но назвать эти прогнозы точными нельзя. У дома 1325 ошибка почти не изменилась: модель по-прежнему оценивает его примерно в 272 тысячи при фактической цене 147 тысяч. Высокое качество этого дома не объясняет его низкую цену; причина остаётся неустановленной.

### Остатки и систематическое смещение

Вспомним знак: **остаток = истинная цена − прогноз**. Положительный остаток означает занижение цены, отрицательный — завышение. Средний остаток по группе помогает увидеть систематическое смещение, а MAE — величину ошибок независимо от знака.

Сравним графики до и после доработки. Группы зададим по известным характеристикам: район, качество и площадь. На одной средней метрике останавливаться не будем.

In [ ]:
validation_view = to_metric_units(validation)
fig, axs = plt.subplots(2, 2, figsize=(13, 8), sharex="col", sharey=True, layout="constrained")
for row, (title, p) in enumerate([("Прежняя модель", old_prediction), ("Выбранная модель", chosen_prediction)]):
    residual = y_val.to_numpy() - p
    for ax, x, label in [(axs[row, 0], p, "Предсказанная цена, $"),
                         (axs[row, 1], validation_view.GrLivArea, "Жилая площадь, м²")]:
        points = ax.scatter(x, residual, c=validation.OverallQual, cmap="viridis", vmin=1, vmax=10, s=22, alpha=.6)
        ax.axhline(0, color="black", linewidth=1)
        ax.set(xlabel=label, ylabel="Остаток, $", title=title)
fig.colorbar(points, ax=axs, label="Общее качество, баллы 1–10", shrink=.8)
save_figure("05_validation_residuals.png")
plt.show()

area_edges = train_view.GrLivArea.quantile([1/3, 2/3]).to_numpy()
def error_table(frame, prediction):
    view = to_metric_units(frame)
    result = view[["Id", "Neighborhood", "GrLivArea", "OverallQual", "YearBuilt", "SalePrice"]].copy()
    result["prediction"] = prediction
    result["residual"] = result.SalePrice - prediction
    result["absolute_error"] = abs(result.residual)
    result["APE_pct"] = 100 * result.absolute_error / result.SalePrice
    result["area_group"] = pd.cut(result.GrLivArea, [-np.inf, *area_edges, np.inf], labels=["Малая", "Средняя", "Большая"])
    result["quality_group"] = np.where(result.OverallQual >= 8, "Качество 8–10", "Качество 1–7")
    return result

val_errors = error_table(validation, chosen_prediction)
for column in ["area_group", "quality_group"]:
    display(val_errors.groupby(column, observed=True).agg(
        n=("Id", "size"), MAE=("absolute_error", "mean"), MAPE_pct=("APE_pct", "mean"),
        mean_residual=("residual", "mean")).round(1))

**Вывод по validation.** Для небольших домов MAE около 9,4 тысячи долларов, для больших — около 24,1 тысячи; MAPE также различается: примерно 7,8% и 10,1%. В группе качества 8–10 средний остаток около **+15,7 тысячи долларов**: систематическое занижение сохраняется. На графике остатков хорошо видны несколько крупных положительных ошибок.

Доработка улучшила общую метрику, но не устранила все недостатки. Логарифмическая целевая переменная меняет веса ошибок при обучении и не гарантирует нулевого среднего остатка в долларах. Следующие возможные эксперименты — уточнение характеристик дорогих объектов и сравнение другого критерия обучения. В рамках этого исследования новые варианты уже не добавляем.

## 7. Какие признаки использует выбранная модель?

Снова применим **перестановочную важность (permutation importance)**. Перемешиваем один исходный столбец validation, пропускаем данные через уже обученный Pipeline и измеряем ухудшение MAPE. Модель при этом не переобучаем.

Одна исходная категория может породить несколько OneHot-столбцов; здесь они проверяются вместе через перемешивание исходного признака. Положительная важность означает ухудшение прогноза после перемешивания. Малое значение не доказывает бесполезности признака: его информация может дублироваться скоррелированными признаками.

Не требуем, чтобы модель обязательно использовала все столбцы. И не называем этот результат причинным влиянием или статистической значимостью.

In [ ]:
importance = permutation_importance(chosen_model, X_val, y_val,
                                  scoring="neg_mean_absolute_percentage_error",
                                  n_repeats=10, random_state=42, n_jobs=1)
importance_table = pd.DataFrame({"Признак": X_val.columns,
    "Рост MAPE, п.п.": 100 * importance.importances_mean,
    "SD перемешиваний, п.п.": 100 * importance.importances_std}).sort_values("Рост MAPE, п.п.", ascending=False)
display(importance_table.head(15).round(3))
importance_table.to_csv(ARTIFACTS / "permutation_importance_validation.csv", index=False)
top = importance_table.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top["Признак"], top["Рост MAPE, п.п."], xerr=top["SD перемешиваний, п.п."], capsize=3)
ax.axvline(0, color="black", linewidth=1)
ax.set(xlabel="Увеличение MAPE после перемешивания, п.п.", ylabel="Исходный признак",
       title="Validation: важность для выбранной модели")
fig.tight_layout()
save_figure("06_importance.png")
plt.show()

## 8. Где нужна дополнительная проверка?

Сформулируем простые **сигналы для проверки нового объекта**. Они используют только признаки и характеристики тренировочной выборки: выход за её диапазоны площади/качества/года, мало домов в районе, редкое сочетание очень большой площади и высокого качества.

В train есть два дома площадью больше 400 м² с качеством 10 и необычно низкой ценой. Это основание выделить такие случаи для дополнительной проверки, а не доказательство, что все большие дома ошибочны. Порог 400 м² выбран по исследованию train; не будем менять его после просмотра test.

Новый объект может быть необычным и внутри всех отдельных диапазонов. Поэтому эти сигналы не описывают всю область применимости. **Наличие флага не означает обязательную ошибку, отсутствие — гарантию точности.** Мы просто проверим качество в обеих группах.

In [ ]:
scope_ranges = {c: (float(train_view[c].min()), float(train_view[c].max()))
                for c in ["GrLivArea", "OverallQual", "YearBuilt"]}
scope_neighborhood_counts = train.Neighborhood.value_counts().to_dict()

def review_flags(frame):
    view = to_metric_units(frame)
    flags = pd.DataFrame(index=frame.index)
    for column, (low, high) in scope_ranges.items():
        flags[f"Вне диапазона {column}"] = view[column].isna() | ~view[column].between(low, high)
    flags["Мало домов в районе"] = frame.Neighborhood.map(scope_neighborhood_counts).fillna(0) < 10
    flags["Очень большой дом высокого качества"] = (view.GrLivArea > 400) & (view.OverallQual >= 8)
    return flags

val_flags = review_flags(validation)
val_errors["review"] = val_flags.any(axis=1)
display(val_errors.groupby("review").agg(n=("Id", "size"), MAPE_pct=("APE_pct", "mean"),
                                           MAE=("absolute_error", "mean")).round(2))
print("True — есть хотя бы один повод для дополнительной проверки.")

На validation флаги выделили только 13 домов. Их MAPE около 8,84%, а у остальных — 8,74%: заметного разделения по относительной ошибке нет. Следовательно, **эти правила пока не установили надёжную границу хороших и плохих прогнозов**. Они полезны как напоминание проверить редкие случаи, но не как автоматическое разрешение на применение модели.

## 9. Фиксируем решение и открываем финальный test

До этого момента test не участвовал ни в выборе признаков, ни в подборе `alpha`, ни в разборе ошибок. Теперь **замораживаем выбранную конфигурацию**, объединяем train и validation и переобучаем всю цепочку. Это позволяет использовать больше наблюдений для оценки весов и статистик препроцессинга.

Правила дополнительной проверки оставляем такими, какими определили их до test. Для сравнения также фиксируем знакомую OLS-модель на шести признаках. Обе модели обучаем на одних и тех же 1168 наблюдениях.

> **Финальная оценка проверяет уже принятое решение.** Если она хуже ожиданий, записываем это в выводах. Не возвращаемся к подбору модели по тому же test, сохраняя за ним название «независимая проверка».

In [ ]:
selected_settings = next(settings for key, _, settings in CANDIDATES if key == SELECTED)
selected_alpha = float(cv_results.loc[SELECTED, "alpha"])
decision = {
    "selected_candidate": SELECTED, "settings": selected_settings, "alpha": selected_alpha,
    "selection_metric": "MAPE", "test_used_for_selection": False,
    "dataset_sha256": dataset_sha256,
    "train_ids": train.Id.tolist(), "validation_ids": validation.Id.tolist(),
    "scope_ranges": scope_ranges, "scope_neighborhood_counts": scope_neighborhood_counts,
    "review_large_area_m2": 400, "review_large_quality": 8,
}
(ARTIFACTS / "decision_before_test.json").write_text(json.dumps(decision, ensure_ascii=False, indent=2), encoding="utf-8")

X_development = train_val.drop(columns=["Id", "SalePrice"])
y_development = train_val.SalePrice
final_model = clone(chosen_model).fit(X_development, y_development)
final_baseline = clone(fitted_models["01"]).fit(X_development, y_development)
print(f"Конфигурация зафиксирована. Ridge alpha={selected_alpha:g}; обучение на {len(train_val)} домах.")

In [ ]:
X_test = test.drop(columns=["Id", "SalePrice"])
y_test = test.SalePrice
test_prediction = final_model.predict(X_test)
baseline_test_prediction = final_baseline.predict(X_test)
final_metrics = pd.DataFrame({
    "Прежняя OLS, 6 признаков": metrics(y_test, baseline_test_prediction),
    "Итоговая модель": metrics(y_test, test_prediction),
}).T
display(final_metrics.round(3))
final_metrics.to_csv(ARTIFACTS / "final_test_metrics.csv")
test_errors = error_table(test, test_prediction)
test_flags = review_flags(test)
test_errors["review"] = test_flags.any(axis=1)
test_errors.to_csv(ARTIFACTS / "test_predictions.csv", index=False)

final_groups = test_errors.groupby("quality_group").agg(
    n=("Id", "size"), MAE=("absolute_error", "mean"), MAPE_pct=("APE_pct", "mean"),
    mean_residual=("residual", "mean"))
display(final_groups.round(2))
display(test_errors.groupby("review").agg(n=("Id", "size"), MAE=("absolute_error", "mean"),
                                         MAPE_pct=("APE_pct", "mean")).round(2))
print(f"Медиана относительной ошибки: {test_errors.APE_pct.median():.1f}%")
print(f"90-й процентиль относительной ошибки: {test_errors.APE_pct.quantile(.9):.1f}%")
print(f"Доля ошибок более 25%: {(test_errors.APE_pct > 25).mean():.1%}")

### Вспомним бутстрап: насколько точна сама оценка ошибки?

Тестовая выборка конечна, поэтому MAPE тоже зависит от того, какие объекты в неё попали. Для уже обученной модели будем выбирать тестовые объекты **с возвращением** и каждый раз пересчитывать среднюю ошибку. Возьмём 2,5-й и 97,5-й процентили полученных значений — знакомый процентильный бутстрап-интервал с номинальным уровнем 95%.

Так мы оцениваем неопределённость средней ошибки при предположении примерно независимых наблюдений из той же совокупности. Это не интервал цены отдельного дома и не полный учёт неопределённости обучения: модель здесь фиксирована. Зависимость соседних домов и перенос на другой период такой простой бутстрап не устраняет.

In [ ]:
rng = np.random.default_rng(42)
bootstrap_indices = rng.integers(0, len(test_errors), size=(2000, len(test_errors)))
bootstrap_mape = test_errors.APE_pct.to_numpy()[bootstrap_indices].mean(axis=1)
bootstrap_mae = test_errors.absolute_error.to_numpy()[bootstrap_indices].mean(axis=1)
error_intervals = pd.DataFrame({
    "Оценка": [test_errors.APE_pct.mean(), test_errors.absolute_error.mean()],
    "Нижняя граница 95%": [np.quantile(bootstrap_mape, .025), np.quantile(bootstrap_mae, .025)],
    "Верхняя граница 95%": [np.quantile(bootstrap_mape, .975), np.quantile(bootstrap_mae, .975)],
}, index=["MAPE, %", "MAE, $"])
display(error_intervals.round(2))
error_intervals.to_csv(ARTIFACTS / "test_error_intervals.csv")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.3))
limit = max(y_test.max(), test_prediction.max())
axs[0].scatter(y_test, test_prediction, s=22, alpha=.6)
axs[0].plot([0, limit], [0, limit], "--", color="black")
axs[0].set(xlabel="Истинная цена, $", ylabel="Предсказанная цена, $", title="Финальный test")
axs[0].locator_params(axis="x", nbins=5)
axs[1].scatter(test_prediction, test_errors.residual, s=22, alpha=.6)
axs[1].axhline(0, color="black", linewidth=1)
axs[1].set(xlabel="Предсказанная цена, $", ylabel="Остаток, $", title="Направление и величина ошибок")
ordered = np.sort(test_errors.APE_pct)
axs[2].step(ordered, np.arange(1, len(ordered)+1)/len(ordered), where="post")
axs[2].axhline(.9, color="black", linestyle="--", linewidth=1)
axs[2].set(xlabel="Абсолютная относительная ошибка, %", ylabel="Доля объектов",
           title="Эмпирическая функция распределения ошибок", ylim=(0, 1.02))
fig.tight_layout()
save_figure("07_final_test.png")
plt.show()

**Как интерпретировать финальный результат.** Сначала сравниваем модели по всем метрикам, затем смотрим на группы и хвост распределения ошибок. Эмпирическая функция распределения показывает долю тестовых домов, ошибка которых не превышает выбранное значение. Это частота на нашей выборке, а не гарантия для следующего дома.

На этом разбиении финальная MAPE получилась **9,35%** вместо **14,64%** у исходной OLS-модели; MAE — около **16,1 тысячи долларов** вместо **24,5 тысячи**. R² вырос примерно с 0,801 до 0,899. Ориентир 8% на финальном test не достигнут.

У 90% тестовых домов относительная ошибка не превышала примерно 18,6%, но у 4,5% домов она больше 25%. Для группы качества 8–10 средний остаток около +16,5 тысячи долларов: ограничение, замеченное на validation, осталось. Флаги дополнительной проверки снова не выделили группу с большей MAPE: у отмеченных домов около 7,9%, у остальных — 9,4%. Поэтому их не следует выдавать за найденные границы надёжности модели.

Небольшая средняя ошибка может сочетаться с несколькими очень неточными прогнозами. Поэтому в записке нужны и MAPE, и ошибки в долларах, и число наблюдений в проблемных группах. Причину каждой отдельной ошибки мы не обязаны знать; непроверенное объяснение обозначаем как гипотезу.

Ниже рассматриваем крупнейшие тестовые ошибки для описания ограничений. Модель после этой таблицы не меняем.

In [ ]:
test_context = test_errors.nlargest(5, "absolute_error").copy()
for column in ["KitchenQual", "BsmtQual", "SaleCondition"]:
    test_context[column] = test.loc[test_context.index, column]
display(test_context[["Id", "Neighborhood", "GrLivArea", "OverallQual", "YearBuilt",
                      "KitchenQual", "SaleCondition", "SalePrice", "prediction", "APE_pct", "review"]].round(2))
test_context.to_csv(ARTIFACTS / "test_largest_errors.csv", index=False)

### Контекст крупнейших тестовых ошибок

Сама строка с дорогим домом ещё не объясняет ошибку. Повторим знакомое сравнение с похожими объектами: ищем до десяти домов из **обучающих 1168 записей**, в том же районе, по стандартизированным площади, качеству, году постройки, площади подвала и гаража. Цену и `Id` в расстояние не включаем. Статистики стандартизации также считаем только по обучающим данным.

Таблица показывает число найденных соседей, медиану и квартили их цен, а также контекст площади и качества. На графиках видно положение ошибочного прогноза. Близость по выбранным признакам не означает одинаковой цены: есть другие характеристики и условия сделки.

In [ ]:
context_columns = ["GrLivArea", "OverallQual", "YearBuilt", "TotalBsmtSF", "GarageArea"]
development_view = to_metric_units(train_val)
test_view = to_metric_units(test)
context_transformer = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
scaled_reference = pd.DataFrame(context_transformer.fit_transform(development_view[context_columns]),
                                index=development_view.index)
scaled_queries = pd.DataFrame(context_transformer.transform(test_view[context_columns]), index=test_view.index)
context_rows = []
fig, axs = plt.subplots(2, 3, figsize=(16, 8))
for ax, (index, obj) in zip(axs.flat, test_context.iterrows()):
    candidates = development_view.loc[development_view.Neighborhood == obj.Neighborhood]
    distances = ((scaled_reference.loc[candidates.index] - scaled_queries.loc[index]) ** 2).sum(axis=1)
    neighbors = candidates.loc[distances.nsmallest(10).index]
    context_rows.append({"Id": int(obj.Id), "Район": obj.Neighborhood, "Всего кандидатов": len(candidates),
        "Соседей": len(neighbors), "Цена объекта, $": obj.SalePrice, "Прогноз, $": obj.prediction,
        "Q1 цены соседей, $": neighbors.SalePrice.quantile(.25),
        "Медиана цены соседей, $": neighbors.SalePrice.median(), "Q3 цены соседей, $": neighbors.SalePrice.quantile(.75),
        "Площадь объекта, м²": obj.GrLivArea, "Медиана площади соседей, м²": neighbors.GrLivArea.median(),
        "Качество объекта": obj.OverallQual, "Медиана качества соседей": neighbors.OverallQual.median()})
    ax.scatter(candidates.GrLivArea, candidates.SalePrice, color="gray", alpha=.3, label="Обучающие дома района")
    ax.scatter(neighbors.GrLivArea, neighbors.SalePrice, color="tab:blue", label="Выбранные соседи")
    ax.scatter(obj.GrLivArea, obj.SalePrice, color="tab:red", marker="*", s=120, label="Истинная цена")
    ax.scatter(obj.GrLivArea, obj.prediction, color="tab:orange", marker="x", s=70, label="Прогноз")
    ax.set(xlabel="Жилая площадь, м²", ylabel="Цена, $", title=f"Id {int(obj.Id)} · {obj.Neighborhood} · n={len(neighbors)}")
handles, labels = axs.flat[0].get_legend_handles_labels()
axs.flat[-1].axis("off")
axs.flat[-1].legend(handles, labels, loc="center", fontsize=10)
fig.tight_layout()
save_figure("08_test_context.png")
plt.show()
test_neighbors = pd.DataFrame(context_rows)
display(test_neighbors.round(1))
test_neighbors.to_csv(ARTIFACTS / "test_neighbor_context.csv", index=False)

## 10. Передаём решение: Pipeline и записка заказчику

Помнишь, зачем трансформеры хранят статистики? Для нового дома нужны **те же правила и обученные преобразования**, а не заново посчитанные средние или категории. Поэтому сохраняем всю цепочку: перевод единиц → признаки → заполнение и кодирование → Ridge → обратное преобразование цены.

В файле используем `cloudpickle`, чтобы сохранить также подготовленные функции и класс трансформера. В отдельном JSON запишем схему входных данных, единицы, версии библиотек и ограничения. Проверим загрузку в новом процессе Python и совпадение прогнозов. Загружать такой файл следует из доверенного источника; формат содержит исполняемые Python-объекты.

Записка заказчику содержит результат и ограничения обычным языком. Таблицы экспериментов, полный аудит признаков и код остаются приложениями.

[Сохранение моделей: варианты и ограничения совместимости](https://scikit-learn.org/stable/model_persistence.html)

In [ ]:
with (ARTIFACTS / "house_price_pipeline.pkl").open("wb") as stream:
    cloudpickle.dump(final_model, stream)
schema = {
    "input_columns": X_development.columns.tolist(),
    "input_dtypes": X_development.dtypes.astype(str).to_dict(),
    "input_units": "Original Ames CSV: areas in square feet, LotFrontage in feet; conversion is inside Pipeline.",
    "missing_values": "Read CSV with keep_default_na=False, na_values=['NA', '']; preserve literal None category.",
    "prediction_units": "US dollars; historical Ames transactions, 2006–2010",
    "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                 "sklearn": sklearn.__version__, "cloudpickle": cloudpickle.__version__},
    "training_rows": len(X_development), "test_rows": len(test),
    "unused_input_columns": ["SaleType", "SaleCondition"],
}
(ARTIFACTS / "input_schema.json").write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding="utf-8")

# Only fixture inputs are serialized here; target values are not required for inference.
fixture = X_test.iloc[:5]
fixture.to_pickle(ARTIFACTS / "inference_check_input.pkl")
check_script = """import sys, cloudpickle, pandas as pd, numpy as np
from pathlib import Path
p = Path(sys.argv[1])
with (p / "house_price_pipeline.pkl").open("rb") as f:
    model = cloudpickle.load(f)
X = pd.read_pickle(p / "inference_check_input.pkl")
np.save(p / "inference_check_prediction.npy", model.predict(X))
"""
subprocess.run([sys.executable, "-c", check_script, str(ARTIFACTS.resolve())], check=True)
np.testing.assert_allclose(np.load(ARTIFACTS / "inference_check_prediction.npy"), final_model.predict(fixture), rtol=1e-10)
print("Модель загружена в новом процессе. Прогнозы совпадают.")

In [ ]:
final_row = final_metrics.loc["Итоговая модель"]
base_row = final_metrics.loc["Прежняя OLS, 6 признаков"]
worst = test_errors.loc[test_errors.absolute_error.idxmax()]
flag_count = int(test_errors.review.sum())
top_features = "; ".join(feature_label(name) for name in importance_table.head(5)["Признак"])
report = f"""# Предварительная оценка цены дома: результаты исследования

## Задача и проверенные данные
Модель оценивает цену продажи дома по характеристикам объекта и дате оценки.
Исследование использует исторические сделки Ames 2006–2010 годов.
После выбора решения обучение выполнено на {len(train_val)} записях.
Окончательная проверка проведена на {len(test)} ранее отложенных записях.
Тип и условия будущей сделки не используются как признаки.

## Полученное качество
- Средняя абсолютная относительная ошибка: **{final_row['MAPE, %']:.2f}%**.
- Средняя абсолютная ошибка: **{final_row['MAE, $']:,.0f} $**.
- RMSE: **{final_row['RMSE, $']:,.0f} $**; R²: **{final_row['R²']:.3f}**.
- У прежней модели на шести признаках MAPE составляла {base_row['MAPE, %']:.2f}%, MAE — {base_row['MAE, $']:,.0f} $.
- У 90% тестовых объектов относительная ошибка не превышала {test_errors.APE_pct.quantile(.9):.1f}%.
- Ошибка более 25% наблюдалась у {(test_errors.APE_pct > 25).mean():.1%} тестовых объектов.
- Приближённый 95%-й бутстрап-интервал средней MAPE для фиксированной модели:
  {error_intervals.loc['MAPE, %', 'Нижняя граница 95%']:.2f}–{error_intervals.loc['MAPE, %', 'Верхняя граница 95%']:.2f}%.
  Он предполагает примерно независимые тестовые наблюдения и не описывает ошибку отдельного дома.

Ориентир 8% MAPE на финальной выборке не достигнут; требования к допустимой ошибке нужно задавать под конкретную задачу.

![Финальная проверка](07_final_test.png)

## Что помогло
В ходе исследования проверены характеристики качества, состояния и устройства дома.
Выбраны логарифмы цены и площадей, Ridge с alpha={selected_alpha:g} и объединение редких категорий.
Более сложные варианты также проверялись; таблицы сравнений приложены.
По проверке на validation наиболее используемые признаки: {top_features}.
Этот результат характеризует модель и не устанавливает причинного влияния.

![Проверка признаков на validation](06_importance.png)

## Ошибки и ограничения
Самая большая абсолютная ошибка на test: объект Id {int(worst.Id)},
фактическая цена {worst.SalePrice:,.0f} $, прогноз {worst.prediction:,.0f} $.
Правила дополнительной проверки сработали для {flag_count} из {len(test)} объектов.
Они отмечают редкие районы, выход за знакомые диапазоны и очень большие дома высокого качества.
Эти правила не гарантируют отсутствие ошибок у остальных домов.
На test отмеченные правилами дома не оказались группой с большей средней относительной ошибкой.
Поэтому флаги служат поводами для проверки данных, а не подтверждённой границей точности.
В группе домов с качеством 8–10 среднее занижение цены составляет
{final_groups.loc['Качество 8–10', 'mean_residual']:,.0f} $; это сохраняющееся ограничение модели.

![Похожие дома и крупнейшие ошибки](08_test_context.png)
В приложении для этих объектов приведены цены и основные характеристики похожих обучающих домов.

Качество проверено для случайно отложенных исторических сделок из того же набора.
Перенос на современный рынок, другой город и будущий период отдельно не проверялся.
Средняя ошибка не является индивидуальным интервалом прогноза.
Цены редких и необычных объектов могут зависеть от сведений, отсутствующих в данных.

## Что передано и что решать дальше
Переданы обученный Pipeline, схема входных данных, версии библиотек,
результаты финального test, графики и таблицы исследования.
Модель подходит как проверенный учебный прототип предварительной оценки в описанной области.
Решение о применении требует сопоставления этих ошибок с допустимой ошибкой конкретной задачи;
единый допустимый порог заказчиком в этой учебной постановке не задан.
Для следующего этапа полезна проверка на более позднем периоде и уточнение данных по крупным ошибкам.
"""
(ARTIFACTS / "customer_note.md").write_text(report, encoding="utf-8")
print(report)

## Приложение. Полный просмотр признаков

В основной истории мы обсудили признаки, которые повлияли на решения. Чтобы остальные не остались за кадром, следующая ячейка сохраняет **по одному графику для каждого из 79 признаков**, используя только train. У категорий показаны медианы, квартили и размеры групп; у числовых признаков — скаттерплоты с ценой. Названия осей и единицы подписаны.

В архив попадут полная таблица признаков, статистики всех категорий и эта галерея. Это материал для дополнительного просмотра, а не требование запомнить 79 названий.

In [ ]:
#@title Сохранить полную галерею признаков в файлы
import textwrap
gallery = ARTIFACTS / "feature_gallery"
gallery.mkdir(exist_ok=True)
for column in X_train:
    if column in categorical_columns:
        table = category_statistics.query("feature == @column").sort_values("median_price")
        fig, ax = plt.subplots(figsize=(8, max(3, len(table) * .25)))
        positions = np.arange(len(table))
        ax.errorbar(table.median_price, positions,
                    xerr=np.vstack([table.median_price-table.q1, table.q3-table.median_price]), fmt="o", capsize=2)
        ax.set_yticks(positions, [f"{r.category} · n={r.n}" for r in table.itertuples()])
        ax.set(xlabel="Цена продажи, $: медиана и интервал Q1–Q3", ylabel="Категория признака")
    else:
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.scatter(train_view[column], train_view.SalePrice, color="black", alpha=.2, s=16)
        unit = ", м²" if column in AREA_COLUMNS else ", м" if column == "LotFrontage" else ", $" if column == "MiscVal" else ""
        ax.set(xlabel=feature_label(column) + unit, ylabel="Цена продажи, $")
    title = f"Train · {column}"
    if column in categorical_columns:
        title += "\n" + textwrap.fill(feature_label(column), width=60)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(gallery / f"{column}.png", dpi=100, bbox_inches="tight")
    plt.close(fig)
print(f"Сохранено графиков: {len(list(gallery.glob('*.png')))}")

archive = Path("ames_final_delivery.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for path in ARTIFACTS.rglob("*"):
        if path.is_file():
            z.write(path, path.relative_to(ARTIFACTS))
print(f"Готов архив: {archive.resolve()}")
try:
    from google.colab import files
    print("Для скачивания выполните: files.download('ames_final_delivery.zip')")
except ImportError:
    print("Локально архив лежит рядом с папкой результатов.")

## Самостоятельно: защитите принятое решение

1. Выберите один эксперимент, который выглядел разумно, но не дал ожидаемого улучшения. Объясните предположение, способ проверки и вывод.
2. Для одного из пяти знакомых домов объясните, что изменилось в прогнозе после добавления признаков. Какие объяснения подтверждены результатом, а какие остаются предположениями?
3. Заказчик предлагает применять модель к новым домам в Бишкеке. Какие результаты нашего исследования можно перенести как метод работы, а что придётся проверить заново?

**Ваши выводы:**

…